# Bubble of Talents - IA Testing Server
## Modelo potente para validación de matching de CVs

Este notebook ejecuta un modelo IA potente en GPU de Colab para testing del sistema de reclutamiento.

**GPU Recomendada**: T4, V100 o A100
**Modelo**: Llama 3.1 8B (optimizado para conversación y análisis)

In [ ]:
# 1. INSTALACIÓN DE DEPENDENCIAS
!pip install transformers torch accelerate bitsandbytes flask pyngrok requests
!pip install huggingface_hub sentence-transformers

In [ ]:
# 2. VERIFICAR GPU DISPONIBLE
import torch
print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU no disponible - usar Runtime > Change runtime type > GPU")

In [ ]:
# 3. CARGAR MODELO OPTIMIZADO
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Configuración para optimizar memoria (cuantización 4-bit)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Modelo recomendado: Microsoft Phi-3-medium (14B, muy eficiente)
model_name = "microsoft/Phi-3-medium-4k-instruct"

print("🔄 Cargando modelo...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

print("✅ Modelo cargado exitosamente")

In [ ]:
# 4. FUNCIONES DE MATCHING
import json
import re
from typing import Dict, List

def generate_cv_matching_score(candidate_data: dict, job_data: dict) -> dict:
    """
    Genera score de matching usando modelo IA potente
    """
    prompt = f"""Eres un experto en reclutamiento. Analiza la compatibilidad entre este candidato y trabajo.

CANDIDATO:
- Nombre: {candidate_data.get('name', 'No especificado')}
- Skills: {', '.join(candidate_data.get('hard_skills', []))}
- Experiencia: {len(candidate_data.get('experience', []))} posiciones anteriores
- Educación: {candidate_data.get('education', [{}])[0].get('title', 'No especificada')}
- Idiomas: {', '.join(candidate_data.get('languages', []))}

TRABAJO:
- Título: {job_data.get('title', '')}
- Nivel: {job_data.get('level', '')}
- Skills requeridas: {', '.join(job_data.get('required_skills', []))}
- Ubicación: {job_data.get('location', '')}

Responde SOLO con este JSON válido:
{{
  "overall_score": [0-100],
  "skills_match": [0-100],
  "experience_match": [0-100],
  "education_match": [0-100],
  "cultural_fit": [0-100],
  "top_strengths": ["strength1", "strength2", "strength3"],
  "concerns": ["concern1", "concern2"],
  "recommendation": "HIRE|INTERVIEW|REJECT",
  "explanation": "Resumen en 2-3 frases"
}}"""

    # Generar respuesta
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=500,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response[len(prompt):].strip()
    
    # Extraer JSON de la respuesta
    try:
        json_match = re.search(r'\{.*\}', response, re.DOTALL)
        if json_match:
            result = json.loads(json_match.group())
            return result
    except:
        pass
    
    # Fallback si no se puede parsear
    return {
        "overall_score": 75,
        "skills_match": 70,
        "experience_match": 80,
        "education_match": 75,
        "cultural_fit": 70,
        "top_strengths": ["Experiencia relevante", "Skills técnicas"],
        "concerns": ["Evaluar fit cultural"],
        "recommendation": "INTERVIEW",
        "explanation": "Candidato prometedor que requiere entrevista para validar fit."
    }

print("✅ Funciones de matching definidas")

In [ ]:
# 5. API SERVER CON FLASK
from flask import Flask, request, jsonify
import threading
import time

app = Flask(__name__)

@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        "status": "healthy",
        "model": model_name,
        "gpu": torch.cuda.is_available(),
        "timestamp": time.time()
    })

@app.route('/match', methods=['POST'])
def match_endpoint():
    try:
        data = request.json
        candidate = data.get('candidate', {})
        job = data.get('job', {})
        
        start_time = time.time()
        result = generate_cv_matching_score(candidate, job)
        processing_time = round((time.time() - start_time) * 1000)
        
        return jsonify({
            "success": True,
            "data": result,
            "meta": {
                "model": model_name,
                "processing_time_ms": processing_time,
                "gpu_used": torch.cuda.is_available()
            }
        })
    except Exception as e:
        return jsonify({
            "success": False,
            "error": str(e)
        }), 500

@app.route('/batch-match', methods=['POST'])
def batch_match_endpoint():
    try:
        data = request.json
        candidates = data.get('candidates', [])
        job = data.get('job', {})
        
        results = []
        start_time = time.time()
        
        for i, candidate in enumerate(candidates):
            match_result = generate_cv_matching_score(candidate, job)
            results.append({
                "candidate_index": i,
                "candidate_id": candidate.get('id'),
                "matching": match_result
            })
        
        # Ordenar por score
        results.sort(key=lambda x: x['matching']['overall_score'], reverse=True)
        
        processing_time = round((time.time() - start_time) * 1000)
        
        return jsonify({
            "success": True,
            "data": {
                "ranked_candidates": results,
                "total_processed": len(candidates)
            },
            "meta": {
                "model": model_name,
                "processing_time_ms": processing_time,
                "avg_time_per_candidate": round(processing_time / len(candidates)) if candidates else 0
            }
        })
    except Exception as e:
        return jsonify({
            "success": False,
            "error": str(e)
        }), 500

print("✅ API Flask configurada")

In [ ]:
# 6. EXPONER API CON NGROK
from pyngrok import ngrok
import threading

# Configurar token de ngrok (opcional, pero recomendado)
# Obtener token gratis en: https://dashboard.ngrok.com/get-started/your-authtoken
# ngrok.set_auth_token("tu_token_aqui")

# Iniciar servidor Flask en thread separado
def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

# Esperar a que Flask inicie
import time
time.sleep(3)

# Crear túnel ngrok
public_url = ngrok.connect(5000)
print(f"🌐 API pública disponible en: {public_url}")
print(f"\n📋 ENDPOINTS DISPONIBLES:")
print(f"   GET  {public_url}/health")
print(f"   POST {public_url}/match")
print(f"   POST {public_url}/batch-match")
print(f"\n💡 Copiar esta URL para configurar en tu backend PHP")
print(f"\n⚠️  IMPORTANTE: Esta URL cambia cada vez que ejecutas el notebook")

In [ ]:
# 7. TEST DE FUNCIONAMIENTO
import requests
import json

# Test data
test_candidate = {
    "id": "test-001",
    "name": "María García",
    "hard_skills": ["PHP", "Laravel", "MySQL", "JavaScript", "Vue.js"],
    "experience": [
        {"title": "Desarrolladora PHP", "company": "TechCorp", "duration": "2 años"},
        {"title": "Desarrolladora Junior", "company": "StartupXYZ", "duration": "1 año"}
    ],
    "education": [{"title": "Ingeniería Informática", "institution": "Universidad XYZ"}],
    "languages": ["Español", "Inglés"]
}

test_job = {
    "title": "Desarrollador PHP Senior",
    "level": "Senior",
    "required_skills": ["PHP", "Laravel", "MySQL", "API REST"],
    "location": "Madrid",
    "experience_required": "3+ años"
}

# Test simple
print("🧪 Ejecutando test de matching...")
test_result = generate_cv_matching_score(test_candidate, test_job)
print(f"\n✅ Resultado del test:")
print(json.dumps(test_result, indent=2, ensure_ascii=False))

print(f"\n📊 Score general: {test_result.get('overall_score', 0)}/100")
print(f"📋 Recomendación: {test_result.get('recommendation', 'N/A')}")

In [ ]:
# 8. MANTENER SERVIDOR ACTIVO
print("🚀 SERVIDOR IA ACTIVO")
print("====================")
print(f"URL pública: {public_url}")
print(f"Modelo: {model_name}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No disponible'}")
print("\n💡 El servidor seguirá activo mientras esta celda esté ejecutándose")
print("💡 Para detener: interrumpir esta celda")
print("\n📝 CONFIGURAR EN BACKEND PHP:")
print(f"   COLAB_API_URL={public_url}")
print(f"   AI_PROVIDER=colab")

# Mantener el servidor corriendo
try:
    while True:
        time.sleep(10)
        # Verificar que Flask sigue corriendo
        try:
            response = requests.get(f"{public_url}/health", timeout=5)
            if response.status_code == 200:
                print(f"🟢 Servidor OK - {time.strftime('%H:%M:%S')}")
            else:
                print(f"🟡 Servidor responde con código {response.status_code}")
        except:
            print(f"🔴 Error de conectividad - {time.strftime('%H:%M:%S')}")
except KeyboardInterrupt:
    print("\n🛑 Servidor detenido por el usuario")
    ngrok.disconnect(public_url)
    print("✅ Túnel ngrok cerrado")